In [21]:
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import requests
import time
import random


In [17]:
headers={
    'User-Agent': 
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

In [ ]:
url = 'http://en.wikipedia.org/wiki/Kevin_Bacon'
req = Request(url, headers={
    'User-Agent': 
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    })
html = urlopen(req)

soup = BeautifulSoup(html, 'html.parser')

for link in soup.find_all('a'):
    if 'href' in link.attrs:
        print(link.attrs['href'])


In [26]:

# Rewrote the code in the book with bs4 instead of urlopen and regex
# this crawls throgh a given wikipedia url and then finds random article links on the, extracts the href and prints it, repeats the step of fetching any random link from this printed page and prints it again, the continues until the loop becomes false.
# I have give it a maximum page of 5 for testing

def getLinks(articleUrl):
    if articleUrl.startswith('//'):
        articleUrl = 'https:' + articleUrl

    response = requests.get(articleUrl, headers=headers)
    time.sleep(random.uniform(1,3))

    soup = BeautifulSoup(response.text, 'html.parser')
    content = soup.find('div', {'id': 'mw-content-text'})
    return content.find_all('a', {'rel':'mw:WikiLink'})

links = getLinks('https://en.wikipedia.org/wiki/Kevin_Bacon')

count = 0
max_pages = 5

while len(links) > 0 and count < max_pages:
    newArticle = links[random.randint(0, len(links)-1)]['href']
    print(newArticle)
    links = getLinks(newArticle)
    count += 1


https://en.wikipedia.org/wiki/JFK_(film)
https://en.wikipedia.org/wiki/Fargo_(1996_film)
https://en.wikipedia.org/wiki/The_Heap_(Fargo)
https://en.wikipedia.org/wiki/Lorne_Malvo
https://en.wikipedia.org/wiki/Serial_killer


In [ ]:
# to check

pages = set()
def getLinks(pageUrl):
    html = urlopen('http://en.wikipedia.org{}'.format(pageUrl))
    bs = BeautifulSoup(html, 'html.parser')
    try:
        print(bs.h1.get_text())
        print(bs.find(id ='mw-content-text').find_all('p')[0])
        print(bs.find(id='ca-edit').find('span')
            .find('a').attrs['href'])
    except AttributeError:
        print('This page is missing something! Continuing.')
        
    for link in bs.find_all('a', href=re.compile('^(/wiki/)')):
        if 'href' in link.attrs:
            if link.attrs['href'] not in pages:
                #We have encountered a new page
                newPage = link.attrs['href']
                print('-'*20)
                print(newPage)
                pages.add(newPage)
                getLinks(newPage)
getLinks('')

In [ ]:
#  checking status code
url = 'https://wikipedia.org'
response = requests.get(url, headers=headers)

print(response)

<Response [200]>


In [ ]:
# to check from above

pages = set()

def getLinks(pageUrl):
    url = f"https://wikipedia.org{pageUrl}"

    try:
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Skipping {pageUrl}: Status {response.status_code}")

        soup = BeautifulSoup(response.text, 'html.parser')

        try:
            print(f"\n[TITLE]: {soup.h1.get_text()}")
        except:
            print("[INFO]: Missing title element.")

        for link in soup.find_all('a'):
            href = link.get('href')

            if href and href.startswith('/wiki/') and ':' not in href:
                if href not in pages:
                    print(f"Found new link: {href}")
                    pages.add(href)

                    time.sleep(1)

                    getLinks(href)

    except requests.exceptions.RequestException as e:
        print(f"Network error: {e}")

In [31]:
# Recursion - I have no idea what it is at this point
# this was also rewritten without regex and urlopen
# this does something similar to the the code above and instead of picking links at randomit uses a recursion instead of a while loop

pages = set()

def getLinks(pageUrl):
    if len(pages) >= 10:
        return

    if pageUrl.startswith('//'):
        pageUrl = 'https:' + pageUrl

    response = requests.get(pageUrl, headers=headers)
    time.sleep(random.uniform(1,3))
    
    soup = BeautifulSoup(response.text, 'html.parser')
    content = soup.find('div', {'id':'mw-content-text'})

    for link in content.find_all('a', {'rel': 'mw:WikiLink'}):
        if len(pages) >= 10:
            break

        href = link.get('href')
        if href and href not in pages:
            print(href)
            pages.add(href)
            getLinks(href)

getLinks('https://en.wikipedia.org')


https://en.wikipedia.org/wiki/Wikipedia
https://en.wikipedia.org/wiki/Main_Page
https://en.wikipedia.org/wiki/Free_content
https://en.wikipedia.org/wiki/Wikipedia:Free_content
https://en.wikipedia.org/wiki/Sexual_objectification#Free_use
https://en.wikipedia.org/wiki/Sex_object_(disambiguation)
//en.wikipedia.org/wiki/Sexual_objectification
https://en.wikipedia.org/wiki/Wikipedia:WikiProject_Countering_systemic_bias
https://en.wikipedia.org/wiki/Special:EditPage/Sexual_objectification
https://en.wikipedia.org/wiki/Talk:Sexual_objectification
